# House Price Prediction - Comprehensive Analysis

This notebook provides a complete analysis of the housing dataset, including data exploration, preprocessing, model training, and visualization.

## Task 1: Load and Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the dataset
df = pd.read_csv('Housing.csv')

print("Dataset successfully loaded!")
print("\n" + "="*80)
print("DATASET SHAPE AND BASIC INFO")
print("="*80)
print(f"\nShape: {df.shape}")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

In [ ]:
# Display first 10 rows
print("\n" + "="*80)
print("FIRST 10 ROWS")
print("="*80)
print(df.head(10))

In [ ]:
# List features and target
print("\n" + "="*80)
print("FEATURES AND TARGET")
print("="*80)

target_col = 'price'
feature_cols = [col for col in df.columns if col != target_col]

print(f"\nTarget Variable: {target_col}")
print(f"\nFeatures ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    dtype = df[col].dtype
    print(f"  {i:2d}. {col:20s} ({dtype})")

In [ ]:
# Check for missing values
print("\n" + "="*80)
print("MISSING VALUES ANALYSIS")
print("="*80)

missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percent.values
})

print("\n", missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0 else "\nNo missing values found!")
print(f"\nTotal Missing Values: {missing_values.sum()}")

In [ ]:
# Dataset statistics
print("\n" + "="*80)
print("DESCRIPTIVE STATISTICS")
print("="*80)
print(df.describe())

## Task 2: Data Cleaning and Preprocessing

In [ ]:
print("\n" + "="*80)
print("PREPROCESSING STEPS")
print("="*80)

# Create a copy for preprocessing
df_processed = df.copy()

# Step 1: Handle missing values
print("\n1. Handling Missing Values...")
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

# Impute numeric columns with median
for col in numeric_cols:
    if df_processed[col].isnull().sum() > 0:
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        print(f"   - Imputed '{col}' with median value: {median_val}")

# Impute categorical columns with mode
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        mode_val = df_processed[col].mode()[0]
        df_processed[col].fillna(mode_val, inplace=True)
        print(f"   - Imputed '{col}' with mode value: {mode_val}")

if df_processed.isnull().sum().sum() == 0:
    print("   ✓ No missing values remaining")

In [ ]:
# Step 2: Remove duplicates
print("\n2. Removing Duplicates...")
initial_rows = len(df_processed)
df_processed = df_processed.drop_duplicates()
removed_duplicates = initial_rows - len(df_processed)

print(f"   - Initial rows: {initial_rows}")
print(f"   - Rows after removing duplicates: {len(df_processed)}")
print(f"   - Duplicates removed: {removed_duplicates}")

In [ ]:
# Step 3: One-hot encode categorical variables
print("\n3. One-Hot Encoding Categorical Variables...")

# Get categorical columns (excluding target)
categorical_features = df_processed.select_dtypes(include=['object']).columns.tolist()

print(f"   - Categorical columns: {categorical_features}")

# One-hot encode
df_encoded = pd.get_dummies(df_processed, columns=categorical_features, drop_first=True)

print(f"   - Shape before encoding: {df_processed.shape}")
print(f"   - Shape after encoding: {df_encoded.shape}")
print(f"   - New features created: {df_encoded.shape[1] - df_processed.shape[1]}")

In [ ]:
# Step 4: Drop irrelevant columns (if any)
print("\n4. Checking for Irrelevant Columns...")

# Identify columns that might not be useful
# In this dataset, all columns are relevant for prediction
print("   - All columns are relevant for price prediction")
print(f"   - Total features for modeling: {len(df_encoded.columns) - 1}")

print("\n" + "="*80)
print("PREPROCESSING COMPLETE")
print("="*80)
print(f"\nFinal dataset shape: {df_encoded.shape}")
print(f"Ready for model training!")

## Task 3: Train and Compare Models

In [ ]:
# Prepare data for modeling
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

# Train-test split
print("\n" + "="*80)
print("DATA SPLITTING")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain-Test Split (80-20):")
print(f"  - Training set size: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Test set size: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"  - Features: {X_train.shape[1]}")

In [ ]:
# Normalize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature Scaling Applied: StandardScaler")

In [ ]:
# Train Linear Regression Model
print("\n" + "="*80)
print("MODEL 1: LINEAR REGRESSION")
print("="*80)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

# Metrics - Training
lr_mae_train = mean_absolute_error(y_train, y_pred_lr_train)
lr_rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_lr_train))
lr_r2_train = r2_score(y_train, y_pred_lr_train)

# Metrics - Testing
lr_mae_test = mean_absolute_error(y_test, y_pred_lr_test)
lr_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_lr_test))
lr_r2_test = r2_score(y_test, y_pred_lr_test)

print("\nTraining Metrics:")
print(f"  - MAE:  ₹{lr_mae_train:,.2f}")
print(f"  - RMSE: ₹{lr_rmse_train:,.2f}")
print(f"  - R²:   {lr_r2_train:.4f}")

print("\nTesting Metrics:")
print(f"  - MAE:  ₹{lr_mae_test:,.2f}")
print(f"  - RMSE: ₹{lr_rmse_test:,.2f}")
print(f"  - R²:   {lr_r2_test:.4f}")

In [ ]:
# Train Random Forest Model
print("\n" + "="*80)
print("MODEL 2: RANDOM FOREST REGRESSOR")
print("="*80)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_rf_train = rf_model.predict(X_train_scaled)
y_pred_rf_test = rf_model.predict(X_test_scaled)

# Metrics - Training
rf_mae_train = mean_absolute_error(y_train, y_pred_rf_train)
rf_rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_rf_train))
rf_r2_train = r2_score(y_train, y_pred_rf_train)

# Metrics - Testing
rf_mae_test = mean_absolute_error(y_test, y_pred_rf_test)
rf_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))
rf_r2_test = r2_score(y_test, y_pred_rf_test)

print("\nTraining Metrics:")
print(f"  - MAE:  ₹{rf_mae_train:,.2f}")
print(f"  - RMSE: ₹{rf_rmse_train:,.2f}")
print(f"  - R²:   {rf_r2_train:.4f}")

print("\nTesting Metrics:")
print(f"  - MAE:  ₹{rf_mae_test:,.2f}")
print(f"  - RMSE: ₹{rf_rmse_test:,.2f}")
print(f"  - R²:   {rf_r2_test:.4f}")

In [ ]:
# Model Comparison
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Train MAE': [f"₹{lr_mae_train:,.2f}", f"₹{rf_mae_train:,.2f}"],
    'Test MAE': [f"₹{lr_mae_test:,.2f}", f"₹{rf_mae_test:,.2f}"],
    'Train RMSE': [f"₹{lr_rmse_train:,.2f}", f"₹{rf_rmse_train:,.2f}"],
    'Test RMSE': [f"₹{lr_rmse_test:,.2f}", f"₹{rf_rmse_test:,.2f}"],
    'Train R²': [f"{lr_r2_train:.4f}", f"{rf_r2_train:.4f}"],
    'Test R²': [f"{lr_r2_test:.4f}", f"{rf_r2_test:.4f}"]
})

print("\n", comparison_df.to_string(index=False))

# Determine better model
print("\n" + "-"*80)
if rf_r2_test > lr_r2_test:
    print("\n✓ Random Forest performs better on test set")
    print(f"  - Higher R² Score: {rf_r2_test:.4f} vs {lr_r2_test:.4f}")
    print(f"  - Lower Test RMSE: ₹{rf_rmse_test:,.2f} vs ₹{lr_rmse_test:,.2f}")
else:
    print("\n✓ Linear Regression performs better on test set")
    print(f"  - Higher R² Score: {lr_r2_test:.4f} vs {rf_r2_test:.4f}")
    print(f"  - Lower Test RMSE: ₹{lr_rmse_test:,.2f} vs ₹{rf_rmse_test:,.2f}")

## Task 4: Visualize Results and Save Charts

In [ ]:
# Chart 1: Price Distribution Histogram
print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.hist(df['price'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Price (₹)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('House Price Distribution', fontsize=14, fontweight='bold', family='Times New Roman')
ax1.grid(axis='y', alpha=0.3)

# Add statistics
mean_price = df['price'].mean()
median_price = df['price'].median()
ax1.axvline(mean_price, color='red', linestyle='--', linewidth=2, label=f'Mean: ₹{mean_price:,.0f}')
ax1.axvline(median_price, color='green', linestyle='--', linewidth=2, label=f'Median: ₹{median_price:,.0f}')
ax1.legend(fontsize=10)

plt.tight_layout()
plt.savefig('price_distribution.png', dpi=300, bbox_inches='tight')
print("\n1. Price Distribution Histogram")
print("   ✓ Saved as: price_distribution.png")
plt.show()

In [ ]:
# Chart 2: Correlation Heatmap
fig2, ax2 = plt.subplots(figsize=(14, 10))

# Calculate correlation matrix
correlation_matrix = df_encoded.corr()

# Create heatmap
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax2)

ax2.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', family='Times New Roman')

plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
print("\n2. Correlation Heatmap")
print("   ✓ Saved as: correlation_heatmap.png")
plt.show()

In [ ]:
# Chart 3: Actual vs Predicted (Random Forest)
fig3, axes = plt.subplots(1, 2, figsize=(15, 6))

# Linear Regression
axes[0].scatter(y_test, y_pred_lr_test, alpha=0.6, color='steelblue', edgecolors='k', s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (₹)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Predicted Price (₹)', fontsize=11, fontweight='bold')
axes[0].set_title('Linear Regression: Actual vs Predicted', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Random Forest
axes[1].scatter(y_test, y_pred_rf_test, alpha=0.6, color='green', edgecolors='k', s=50)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price (₹)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Predicted Price (₹)', fontsize=11, fontweight='bold')
axes[1].set_title('Random Forest: Actual vs Predicted', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300, bbox_inches='tight')
print("\n3. Actual vs Predicted Comparison")
print("   ✓ Saved as: actual_vs_predicted.png")
plt.show()

In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated Files:")
print("  1. price_distribution.png")
print("  2. correlation_heatmap.png")
print("  3. actual_vs_predicted.png")
print("\nAll visualizations have been saved successfully!")